# A3-M2 · FAISS 기초
교안의 같은 번호 단원과 함께 봅니다.

## 01-1. 준비 — 설치 확인

In [ ]:
pip install faiss-cpu numpy pandas

In [5]:
# 터미널에서 먼저: pip install faiss-cpu numpy pandas
import faiss
import numpy as np
import pandas as pd
print("faiss", faiss.__version__, "/ numpy", np.__version__)

faiss 1.15.1 / numpy 2.4.6


## 01-2. 손으로 계산 — pandas로 L2 거리 구하기

In [6]:
query_vector = np.array([[3]], dtype=np.float32)      # 3과 가장 가까운 값을 찾고 싶다
db_vectors = np.array([[1.], [2.], [3.], [4.], [5.]], dtype=np.float32)

df = pd.DataFrame(db_vectors[:, 0], columns=["db_vectors"])
df["query"] = query_vector[0][0]
df["L2"] = (df["query"] - df["db_vectors"]) ** 2        # 차이의 제곱
print(df)
print("▶ 거리 순으로 정렬")
print(df.sort_values(by="L2"))

   db_vectors  query   L2
0         1.0    3.0  4.0
1         2.0    3.0  1.0
2         3.0    3.0  0.0
3         4.0    3.0  1.0
4         5.0    3.0  4.0
▶ 거리 순으로 정렬
   db_vectors  query   L2
2         3.0    3.0  0.0
1         2.0    3.0  1.0
3         4.0    3.0  1.0
0         1.0    3.0  4.0
4         5.0    3.0  4.0


## 01-3. FAISS로 같은 계산 — IndexFlatL2

In [7]:
database_size, dimension = db_vectors.shape            # (5, 1) → 5개, 1차원
index = faiss.IndexFlatL2(dimension)                     # 1차원 벡터용 '전수 비교' 인덱스
index.add(db_vectors)                                    # 벡터 5개를 넣는다
print("ntotal:", index.ntotal, "| d:", index.d, "| 학습 필요 없음:", index.is_trained)

k = 5                                                    # 가까운 것 5개
distances, indices = index.search(query_vector, k)
print("distances:", distances)
print("indices  :", indices)
print(distances.dtype, indices.dtype, distances.shape)

ntotal: 5 | d: 1 | 학습 필요 없음: True
distances: [[0. 1. 1. 4. 4.]]
indices  : [[2 1 3 0 4]]
float32 int64 (1, 5)


## 01-4. k가 데이터 개수보다 크면?

In [8]:
distances, indices = index.search(query_vector, 7)      # 5개뿐인데 7개를 달라고 하면
print(distances)
print(indices)
print("float32 최댓값:", np.finfo(np.float32).max)

[[0.0000000e+00 1.0000000e+00 1.0000000e+00 4.0000000e+00 4.0000000e+00
  3.4028235e+38 3.4028235e+38]]
[[ 2  1  3  0  4 -1 -1]]
float32 최댓값: 3.4028235e+38


## 01-5. 2차원 벡터와 여러 질문을 한 번에

In [9]:
query2 = np.array([[3, 2]], dtype=np.float32)
db2 = np.array([[1, 2], [2, 8], [3, 9], [4, 3], [5, 1]], dtype=np.float32)

df2 = pd.DataFrame(db2, columns=["x1", "x2"])
df2["L2"] = (query2[0][0] - df2["x1"]) ** 2 + (query2[0][1] - df2["x2"]) ** 2
print(df2.sort_values(by="L2"))

index2 = faiss.IndexFlatL2(2)
index2.add(db2)
print(index2.search(query2, 5))

queries = np.array([[3, 2], [2, 9]], dtype=np.float32)   # 질문 2개를 한 번에
D, I = index2.search(queries, 2)
print("D =", D)
print("I =", I)

    x1   x2    L2
3  4.0  3.0   2.0
0  1.0  2.0   4.0
4  5.0  1.0   5.0
1  2.0  8.0  37.0
2  3.0  9.0  49.0
(array([[ 2.,  4.,  5., 37., 49.]], dtype=float32), array([[3, 0, 4, 1, 2]]))
D = [[2. 4.]
 [1. 1.]]
I = [[3 0]
 [1 2]]


## 02-1. 내적(IP)과 코사인 유사도

In [10]:
ip = faiss.IndexFlatIP(2)                                # 내적: 클수록 가깝다
ip.add(db2)
print("내적  :", ip.search(query2, 3))

db_n = db2.copy()                                        # 원본을 지키려고 복사
q_n = query2.copy()
faiss.normalize_L2(db_n)                                 # 제자리에서 길이 1로 (반환값 None)
faiss.normalize_L2(q_n)
print("길이:", np.linalg.norm(db_n, axis=1).round(3))

cos = faiss.IndexFlatIP(2)
cos.add(db_n)
print("코사인:", cos.search(q_n, 3))

내적  : (array([[27., 22., 18.]], dtype=float32), array([[2, 1, 3]]))
길이: [1. 1. 1. 1. 1.]
코사인: (array([[0.9984604, 0.9246781, 0.8682431]], dtype=float32), array([[3, 4, 0]]))


## 02-2. 꺼내 보기·비우기·자료형

In [11]:
print("3번 벡터:", index2.reconstruct(3))
print("0~1번  :", index2.reconstruct_n(0, 2))

try:
    index2.add(np.array([1, 2], dtype=np.float32))      # 1차원 배열(모양 (2,))을 넣으면
except Exception as e:
    print("오류:", type(e).__name__, "-", e)

index2.reset()
print("reset 후 ntotal:", index2.ntotal)

3번 벡터: [4. 3.]
0~1번  : [[1. 2.]
 [2. 8.]]
오류: ValueError - not enough values to unpack (expected 2, got 1)
reset 후 ntotal: 0


## 03-1. ID 붙이기 — IndexIDMap

In [12]:
rng = np.random.default_rng(0)
xb = rng.random((6, 4)).astype("float32")                # 4차원 벡터 6개
ids = np.array([101, 102, 103, 104, 105, 106], dtype="int64")

idmap = faiss.IndexIDMap(faiss.IndexFlatL2(4))
idmap.add_with_ids(xb, ids)
D, I = idmap.search(xb[:1], 3)
print("돌려받는 번호:", I)

try:
    idmap.add(xb[:1])                                    # ID 없이 add 하면
except Exception as e:
    print("오류:", type(e).__name__)

removed = idmap.remove_ids(np.array([102, 104], dtype="int64"))
print("지운 개수:", removed, "| 남은 개수:", idmap.ntotal)
print("남은 ID:", faiss.vector_to_array(idmap.id_map))

돌려받는 번호: [[101 105 104]]
오류: RuntimeError
지운 개수: 2 | 남은 개수: 4
남은 ID: [101 103 105 106]


## 03-2. ID로 꺼내기·조건 걸어 검색하기

In [13]:
idmap2 = faiss.IndexIDMap2(faiss.IndexFlatL2(4))         # ID로 꺼내기까지 지원
idmap2.add_with_ids(xb, ids)
print("103번 벡터:", idmap2.reconstruct(103))

docs = {101: "배송", 102: "배송", 103: "결제", 104: "환불", 105: "환불", 106: "회원"}
allowed = [i for i, cat in docs.items() if cat == "환불"]  # 환불 문서만
sel = faiss.IDSelectorBatch(np.array(allowed, dtype="int64"))
params = faiss.SearchParameters(sel=sel)
print("환불 문서 안에서만:", idmap2.search(xb[:1], 2, params=params))

103번 벡터: [0.543625   0.9350724  0.81585354 0.0027385 ]
환불 문서 안에서만: (array([[0.35689244, 0.60399044]], dtype=float32), array([[105, 104]]))
